In [1]:
# Runtime evidence: this cell checks the environment actually used by the kernel.
import sys as _shape_sys, json as _shape_json, importlib.util as _shape_util
print("SHAPE_RUNTIME_PROVENANCE=" + _shape_json.dumps({
    "python_executable": _shape_sys.executable,
    "python_version": _shape_sys.version.split()[0],
    "shape_lab_origin": _shape_util.find_spec("shape_lab").origin
}))
del _shape_sys, _shape_json, _shape_util


SHAPE_RUNTIME_PROVENANCE={"python_executable": "/opt/pyvenv/bin/python", "python_version": "3.13.5", "shape_lab_origin": "/tmp/shape-course-q90rdcpz/course/src/shape_lab/__init__.py"}


# Stage 08 consolidation — Read a barcode as a history, not a collection of scores

Read `sessions/08_session.md` first. This is an executed reference, not an assessment of you.

**Evidence:** Real digit intensity filtration and an exact complete/truncated square control.

In [2]:
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib','inline')
from pathlib import Path
import sys, json
import numpy as np

ROOT = Path.cwd().resolve()
while not (ROOT/'data/raw/iris.csv').is_file():
    if ROOT == ROOT.parent:
        raise FileNotFoundError('Start Jupyter in the extracted listening_to_shape_lab folder.')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT/'src'))


## Step 1

Recover the exact square interval with the course implementation.

Before running: write your expected result and name one way this check could fail.

In [3]:
from shape_lab.persistence import rips_filtration, persistent_homology, diagram, pixel_filtration
square = np.array([[0.,0.],[1.,0.],[1.,1.],[0.,1.]])
bars = persistent_homology(rips_filtration(square))
h1 = diagram(bars, 1)
print(h1)
assert np.allclose(h1, [[1., np.sqrt(2)]])

[[1.         1.41421356]]


## Step 2

Read barcode counts at declared thresholds on a real digit and compare with the direct square-cell oracle.

Before running: write your expected result and name one way this check could fail.

In [4]:
from shape_lab.data import digit_example
from shape_lab.audits import pixel_betti_oracle
image, sample_id = digit_example(0)
f = 1-image/16.
bars = persistent_homology(pixel_filtration(f))
counts = []
for t in [0., .25, .5, .75, 1.]:
    predicted = [sum(b.dim==k and b.birth<=t<b.death for b in bars) for k in [0,1]]
    direct = list(pixel_betti_oracle(f<=t))
    assert predicted == direct
    counts.append([t, *direct])
print(counts)

[[0.0, 0, 0], [0.25, 4, 0], [0.5, 1, 1], [0.75, 1, 1], [1.0, 1, 0]]


## Step 3

Stop the square early to demonstrate an infinite endpoint caused by truncation.

Before running: write your expected result and name one way this check could fail.

In [5]:
truncated = diagram(persistent_homology(rips_filtration(square, max_edge=1.2)), 1)
assert len(truncated)==1 and np.isinf(truncated[0,1])
result = {'sample_id': sample_id, 'image_counts': counts,
          'square_complete_h1': h1.tolist(), 'truncated_square_h1_survives_end': True,
          'truncated_max_edge': 1.2, 'global_essential_claim': False}

## Keep execution evidence separate from learning evidence

The following saves this reference calculation. Write your own explanation in `my_work/`; no learner score is assigned.

In [6]:
result['stage'] = 8
result['execution_role'] = 'reference_consolidation'
result['data_role'] = 'Real digit intensity filtration and an exact complete/truncated square control.'
output = ROOT/'reports/consolidation_v4/08.json'
output.parent.mkdir(parents=True, exist_ok=True)
output.write_text(json.dumps(result, indent=2, allow_nan=False) + '\n')
print('Saved reference result:', output.relative_to(ROOT))


Saved reference result: reports/consolidation_v4/08.json
